# Superstore Sales & Profitability Analysis | Python

**Business question:** Which categories, regions, and customer segments drive sales and profit, and where should the business investigate losses?

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn. Data source: `Sample - Superstore.xlsx` (Orders sheet).

### 1. Import and understand the data ###

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read the Orders sheet from the Excel file
df = pd.read_excel("Sample - Superstore.xlsx", sheet_name="Orders")
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

**Data quality note:** The Orders sheet contains 10,194 rows, with no missing values or exact duplicate rows in the original file. We inspect sales outliers rather than deleting them automatically because large orders may be valid.

### 2. Data Cleaning ###

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Check text columns before cleaning
text_columns = df.select_dtypes(include="object").columns
print("Text columns:", list(text_columns))

### outlier

In [ ]:
Q1 = df["Sales"].quantile(0.25)
Q3= df["Sales"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

sales_outliers = df [
    (df["Sales"] < lower_bound) |
    (df["Sales"] > upper_bound)
]

sales_outliers.shape

In [ ]:
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Number of Sales Outliers:", len(sales_outliers))

# in sales if there is one large deal it is normal and could happen so i will leave it

### feature engineering

In [ ]:
df["Profit Margin"] = (df["Profit"] / df["Sales"]) * 100

In [ ]:
df["Shipping Duration"] = (df["Ship Date"] - df["Order Date"]).dt.days

In [ ]:
def sales_category(sales):
    if sales < 100:
        return "Low"
    elif sales < 500:
        return "Medium"
    else:
        return "High"

df["Sales Performance Category"] = df["Sales"].apply(sales_category)

In [ ]:
df[
    ["Sales", "Profit", "Profit Margin",
     "Shipping Duration", "Sales Performance Category"]
].head()

### reusable preprocessing pipelines

In [ ]:
class DataCleaner:
    def __init__(self, data):
        self.data = data.copy()

    def clean(self):
        self.data = self.data.drop_duplicates()
        for col in self.data.select_dtypes(include="object").columns:
            self.data[col] = self.data[col].str.strip()
        return self.data


In [ ]:
cleaner = DataCleaner(df)
clean_df = cleaner.clean()
print("Rows after cleaning:", len(clean_df))

# Use the cleaned dataset in the rest of the analysis
df = clean_df.copy()


## exploratory data analysis

 Total Sales by category

In [ ]:
summary = df.groupby("Category", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x="Category", y="Sales", color="steelblue")
plt.title("Total Sales by Category")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Total profit by category

In [ ]:
summary = df.groupby("Category", as_index=False)["Profit"].sum().sort_values("Profit", ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x="Category", y="Profit", color="steelblue")
plt.title("Total Profit by Category")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Sales by region

In [ ]:
summary = df.groupby("Region", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x="Region", y="Sales", color="steelblue")
plt.title("Total Sales by Region")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Sales over time

In [ ]:
monthly_sales = df.groupby(df["Order Date"].dt.to_period("M"))["Sales"].sum()
monthly_sales.index = monthly_sales.index.to_timestamp()
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales.index, monthly_sales.values)
plt.title("Monthly Sales Trend")
plt.xlabel("Order Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Sales vs Profit

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(df["Sales"], df["Profit"])

plt.title("Sales vs Profit")
plt.xlabel("Sales")
plt.ylabel("Profit")

plt.show()

Sales by segment

In [ ]:
summary = df.groupby("Segment", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=summary, x="Segment", y="Sales", color="steelblue")
plt.title("Total Sales by Segment")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

Total profit by sub-category

In [ ]:
summary = df.groupby("Sub-Category", as_index=False)["Profit"].sum().sort_values("Profit", ascending=True)
plt.figure(figsize=(12, 5))
sns.barplot(data=summary, x="Sub-Category", y="Profit", color="steelblue")
plt.title("Total Profit by Sub-Category")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()
plt.show()

Discount vs Profit

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(df["Discount"], df["Profit"])

plt.title("Discount vs Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")

plt.show()

## Correlation Analysis

In [ ]:
num_data = df.select_dtypes(include="number")
corr = num_data.corr()

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True)

plt.title("Correlation Matrix")
plt.show()

## Distribution analysis

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df["Sales"], bins=30)

plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")

plt.show()

## Memory usage

In [ ]:
memory_before = df.memory_usage(deep=True).sum()

for col in ["Category", "Segment", "Region", "Ship Mode"]:
    df[col] = df[col].astype("category")

memory_after = df.memory_usage(deep=True).sum()

print("Memory before:", memory_before)
print("Memory after:", memory_after)

The correlation matrix helps us compare numeric variables. Correlation does not prove that one variable causes another. In this dataset, discounts have a negative relationship with profit, which is worth investigating by category and product.

### Key Findings

- Technology has the highest category sales ($839,893.28) and profit ($146,543.38).
- The West region has the highest sales ($739,813.61).
- Consumer has the highest segment sales ($1,170,659.79).
- Tables (-$17,753.21), Bookcases (-$3,632.07), and Supplies (-$1,171.39) have negative total profit.
- Discounts and profit have a negative correlation (about -0.22), but this alone does not establish causation.

### Conclusion

Total sales are strong, but profitability varies across product categories. Furniture generates substantial sales while contributing much less profit than Technology or Office Supplies. I would review discounting and product costs for loss-making sub-categories before recommending pricing changes.

### Automated KPI Summary

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
average_sales = df["Sales"].mean()
average_profit = df["Profit"].mean()
profit_margin = (total_profit / total_sales) * 100

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")
print(f"Average Sales per Row: ${average_sales:,.2f}")
print(f"Average Profit per Row: ${average_profit:,.2f}")

### Automated Analytical Summary

In [ ]:
top_category = df.groupby("Category")["Sales"].sum().idxmax()
top_region = df.groupby("Region")["Sales"].sum().idxmax()
top_segment = df.groupby("Segment")["Sales"].sum().idxmax()

print("Top Sales Category:", top_category)
print("Top Sales Region:", top_region)
print("Top Sales Segment:", top_segment)

### Export Cleaned Data

In [ ]:
df.to_csv("cleaned_superstore.csv", index=False)
print("Cleaned data exported successfully.")

### Final Summary

I cleaned and analyzed 10,194 Superstore order lines using Python. Total sales were $2,326,534.35 and total profit was $292,296.81, giving a 12.56% profit margin. The main opportunity is to investigate low-profit furniture products and discounting while protecting strong-performing categories.